# Emergency Department (ED) Discrete Event Simulation

**1.a Arrival → {Immediate Care Required} → Trauma Room + Care**

- Arrival
    - $t_{a}$: Patient arrives Exponentially Distributed w/ $\mu$ = 10 Minutes
    - About 10% of Arrivals REQUIRE Immediate Care
- T: Number of Trauma / Immediate Care Rooms
    - As long as T > 0, the patient will immediately be placed in a trauma room.
    - If T = 0, then the patient will be assigned to the soonest available trauma or non-trauma room (T or E).

**1.b Arrival → {Non-Immediate Care} → Triage** 

- Arrival
    - $t_{a}$: Patient arrives Exponentially Distributed w/ $\mu$ = 10 Minutes
- {Non-Immediate Care}
    - About 90% of Arrivals Do NOT Require Immediate Care
    - {Add to $Q_t$}
- Join Triage Queue ($Q_t$: Queue for Triage)
- Start Triage
    - When N > 0: If a Nurse is Available, Start Triage
    - {Subtract from $Q_t$}
    - {Subtract from N} (N: Available Nurses)
    - The Triage Service Time is Exponentially Distributed with Rate $\mu$ = 12 Patients per Hour.
- End Triage
    - After Triage is Complete, {Add to N}
    - The patient moved to the Exam Room Queue, $Q_E$ {Add to $Q_E$}

**2.1 Exam Room Queue → Exam Room → {Testing REQUIRED} → Doctor Consultation Queue**

- Exam Room Queue, $Q_E$
    - The Exam Room Queue time is Exponentially Distributed with a Rate $\mu$ = 8 Patients per Hour.
    - When E > 0: If an Exam Room is Available, Patient Treatment Begins
- Patient Treatment {Testing Needed}
    - About 65% of Patient REQUIRE some form of Imaging/Labs During Their Emergency Department Visit
    - {Add to $Q_I$}, $Q_I$: Queue to have imaging or labs completed.

**2.2 Exam Room Queue → Exam Room → {NO Testing} → Doctor Consultation Queue**

- Exam Room Queue, $Q_E$
    - The Exam Room Queue time is Exponentially Distributed with a Rate $\mu$ = 8 Patients per Hour.
    - When E > 0: If an Exam Room is Available, Patient Treatment Begins
- Join Doctor Consultation Queue, $Q_D$
    - The 45% of Patients that Don’t Require Imaging/Lab Testing will Immediately Join the Doctor Consultation Queue.
    - The Patient Joins the Doctor Consultation Queue, {Add to $Q_D$}

**2.3 Trauma → {Testing REQUIRED} → Doctor Consultation Queue**

- Trauma Patient Needing Testing
    - 75% of Trauma Patient Will Requiring Testing
    - {Add to $Q_I$}
- Start Imaging/Lab Testing
    - I > 0: When the Imaging/Lab Testing is Table, The Imaging/Labs will begin.
    - {Subtract from I} (I: Imaging/Lab Rooms Available)
    - The Imaging/Labs Time is Exponentially Distributed with $\mu$ = 4 Patients per Hour.
- End Imaging/Lab Testing → Enter Doctor Consultation Queue, $Q_D$
    - After Imaging/Lab Testing, {Add to I}
    - The Patient Joins the Doctor Consultation Queue, {Add to $Q_D$}

**2.4 Trauma → {NO Testing} → Doctor Consultation Queue**

- Trauma Patient Not Requiring Testing Will Immediately Join Doctor Consultation Queue (25% of Trauma Patients)
    - {Add to $Q_D$}

**3.1 Doctor Consultation Queue → Doctor Consultation → Addition Care Assessment**

- Doctor Consultation Start
    - When a Doctor Becomes Available, D > 0, a Roomed Patient will be Seen
    - {Subtract from $Q_D$}
    - {Subtract from D}
- Doctor Consultation Time Exponentially Distributed with Rate $\mu$ = 3 Patients per Hour
- Doctor Consultation Finished and Additional Care Assessment Made
    - Doctor Consultation will Determine Patient’s Next Step:
        - 25% Admitted to Hospital, 75% Discharged
    - {Add to D}

In [3]:
# Import the necessary libraries to create the simulation
import numpy as np
import pandas as pd
import simpy
import random
import datetime

# Define the constants for the simulation
mean_arrival_time = 10 # Mean time between arrivals
trauma_probability = 0.1 # Probability of a trauma patient
triage_rate = 12 # Rate at which patients are triaged per hour
testing_probability = 0.65 # Probability of a patient requiring testing
testing_rate = 8 # Rate at which patients are tested per hour
admission_probability = 0.25 # Probability of a patient requiring admission
discharge_probability = 0.75 # Probability of a patient being discharged

In [5]:
# Define the Emergency Department simulation environment
class EmergencyDepartment:
    def __init__(self, env):
        self.env = env
        
        # Resources for the Emergency Department
        self.trauma_rooms = simpy.Resource(env, capacity=1)
        self.triage_rooms = simpy.Resource(env, capacity=1)
        self.nurses = simpy.Resource(env, capacity=1)
        self.testing_rooms = simpy.Resource(env, capacity=1)
        self.doctors = simpy.Resource(env, capacity=1)
        
        # Queues for the Emergency Department
        self.triage_queue = simpy.PriorityResource(env)
        self.exam_queue = simpy.PriorityResource(env)
        self.testing_queue = simpy.PriorityResource(env)
        self.doctor_consult_queue = simpy.PriorityResource(env)
        
        # Variables to track the number of patients in the Emergency Department
        self.patients_waiting_triage = 0
        self.patients_waiting_trauma = 0
        self.patients_waiting_exam = 0
        self.patients_waiting_testing = 0
        self.patients_waiting_doctor = 0
        self.patients_waiting_admission = 0
        self.patients_waiting_discharge = 0
        
    # Define the patient triage process
    # The patient can only be triage if a nurse is available to triage the patient
    def traige(self, patient):
        # Simulate the triage process
        with self.triage_room.request() as request:
            yield request
            # Triage the patient
            triage_time = random.expovariate(1 / (60 / triage_rate)) # Convert triage rate to minutes
            yield self.env.timeout(triage_time)
            # Print the triage time
            print(f"Patient {patient} triaged at {self.env.now}")

    # Define the patient testing (imaging/lab) process
    # The patient can only be tested if they are in a trauma or exam room and if the testing room is available
    def test(self, patient):
        # Simulate the testing process
        with self.testing_rooms.request() as request:
            yield request
            # Test the patient
            test_time = random.expovariate(1 / (60 / testing_rate)) # Convert testing rate to minutes
            yield self.env.timeout(test_time)
            # Print the testing time
            print(f"Patient {patient} tested at {self.env.now}")
            

                
            

        
        
        
        

In [ ]:
    # Define the patient arrival process
    def patient_arrival(self):
        # Simulate the arrival process
        # 10% of patients are trauma patients
        if random.random() < trauma_probability:
            patient = Patient(self.env, self, trauma=True)
        else:
            patient = Patient(self.env, self, trauma=False)
        # If the patient is a trauma patient, they are sent directly to a trauma room
        if patient.trauma:
            self.env.process(patient.trauma_room())
        # If the patient is not a trauma patient, then they are sent to the triage process (join the triage queue)
        else:
            self.env.process(patient.triage())
            
        # Print when the patient arrives and the patient's type
        print(f'{self.env.now:.2f}: Patient {patient.id} arrives ({patient.type})')
        
        # Wait for the next patient arrival
        interarrival_time = random.expovariate(1 / mean_arrival_time)
        yield self.env.timeout(interarrival_time)
        # Start the next patient arrival
        self.env.process(self.patient_arrival())